# 01 · Build Pipeline & Discovery Cold-Start

**This notebook does two things:**

1. Runs `rca build` to generate taxonomy JSONs, module `.py` files,
   `pipeline_config.json`, and `build_manifest.json`.
2. Optionally runs the **discovery cold-start** if you have no taxonomy yet.

**New in this version — Enhancement module setup:**
3. Configures the **feedback store** for human-in-the-loop corrections
4. Validates **language detection** readiness
5. Reviews **cost tracking** and **quality dashboard** settings

After this notebook → run **`02_classification.ipynb`**.

## 0 · Imports

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
import sys, os
from pathlib import Path

# Ensure project root is on sys.path and is the working directory
_root = Path().absolute()
# Navigate up until we find the rca_pipeline package
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import subprocess, sys, json, asyncio
from pathlib import Path
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

print("Python", sys.version.split()[0])
print("nest_asyncio applied ✓")

## 1 · Paths & config

In [ ]:
# ── Paths ───────────────────────────────────────────────
DEFINITION_FILE = "definitions\scopus_content_pipeline_definition_v2.xlsx"
TAXONOMY_DIR    = "taxonomies_scopus_content"
MODULE_DIR      = "generated/modules"
CONFIG_FILE     = "generated/pipeline_config.json"
MANIFEST_FILE   = "generated/build_manifest.json"

# ── Data (used only for discovery) ─────────────────────────────
DATA_FILE = "data\Scopus_Content_Collapsed.xlsx"
ID_COL    = "Reference #"
TEXT_COL  = "Conversation"

# ── LLM ─────────────────────────────────────────────────
MODEL       = "azure/gpt-5.4-mini"
CONCURRENCY = 4
RETRIES     = 3

# ── Enhancement directories ──────────────────────────────────
FEEDBACK_DIR     = "runs/feedback"
COST_REPORT_DIR  = "runs/cost_reports"
QUALITY_DIR      = "runs/output"

## 2 · Run `rca build`

Generates taxonomy JSONs, missing `.py` files, `pipeline_config.json`, and `build_manifest.json`.

### Policy taxonomy harmonisation — CAR Manual v3.37 + Content Coverage Guide (June 2026)

**New policy codes added:**

| Code | Label | Source |
|---|---|---|
| PA10 | Article-in-Press (AiP) eligibility and processing | CAR Manual §5 |
| PA11 | Excluded item types (full CAR §1 exclusion list) | CAR Manual §1 |
| PU2.x | Customer root cause | Merged from policy_understanding into policy_compliance |
| CR1.8 | AiP visibility or missing (contact reason) | CAR Manual §5 |

**Modules changed:**

- `policy_compliance`: expanded with PA10/PA11 + customer understanding fields (`customer_understanding_issue`, `customer_root_cause_code`)
- `policy_understanding`: disabled (fields merged into policy_compliance)

After updating the Excel, run Section 2 (force-rebuild) to regenerate `policy_compliance.json` and `policy_compliance.py`.  
Use `rca build build --force policy_compliance` to regenerate only the compliance module.


### PA12–PA17 additions (June 2026) — first classification run required

**New policy codes added to `policy_compliance` module:**

| Code | Label | Source |
|---|---|---|
| PA12 | Patent indexing and coverage | Content Coverage Guide §2.3 |
| PA13 | Author identity and profile management | Content Coverage Guide §3.3 |
| PA14 | Affiliation profile curation | Content Coverage Guide §3.4 |
| PA15 | Open Access classification | Content Coverage Guide §3.1 |
| PA16 | Errata and record corrections | CCG §3.1 + CAR Manual §6 |
| PA17 | Cited references coverage and linkage | Content Coverage Guide §3.2, §4.5 |

After updating the Excel, rebuild the module:
`rca build build --force policy_compliance`

Then re-run classification (notebook 12) to produce a new `Scopus_Content_Policy_enriched.xlsx` with PA12–PA17 tickets included. The R analysis script (`scopus_effort_master_v1.R`) will automatically handle the new codes — `policy_doc_map` already contains entries for PA12–PA17.


In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "build",
     "--input",        DEFINITION_FILE,
     "--taxonomy-dir", TAXONOMY_DIR,
     "--module-dir",   MODULE_DIR,
     "--config-file",  CONFIG_FILE,
     "--manifest-file",MANIFEST_FILE,
     "--force-all",  # ← uncomment to regenerate ALL .py files
    ],
    capture_output=True, text=True, encoding="utf-8", errors="replace", cwd="."
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("rca build failed")

In [ ]:
import re

bp_src  = Path(__import__('rca_pipeline.build', fromlist=['_']).__file__).read_text(encoding='utf-8')
m       = re.search(r'_GENERATOR_VERSION\s*=\s*["\']([^"\']+)["\']', bp_src)
current = m.group(1) if m else 'unknown'

if Path(MANIFEST_FILE).exists():
    manifest = json.load(open(MANIFEST_FILE))
    stored   = manifest.get('generator_version', '0.0.0')
    if stored != current:
        print(f'⚠️  Manifest version {stored} != generator {current}')
        print(f'   → Re-run with --force-all to regenerate.')
    else:
        print(f'✓ Generator v{current} — all files are current')
        for mod, info in manifest.get('modules', {}).items():
            py_ok = (Path(MODULE_DIR) / f"{mod}.py").exists()
            gen   = info.get('py_generated_at', 'never')[:19]
            print(f'  {mod:25s}  generated {gen}  file={"OK" if py_ok else "MISSING"}')
else:
    print(f'ℹ️  No manifest yet (v{current}) — first run will generate all files')

## 3 · Review what was generated

In [ ]:
manifest = json.load(open(MANIFEST_FILE)) if Path(MANIFEST_FILE).exists() else {}
print(f"Manifest updated: {manifest.get('generated_at', 'n/a')}")
print()

# Module .py files
print("Module .py files:")
cfg_data = json.load(open(CONFIG_FILE)) if Path(CONFIG_FILE).exists() else {"modules": {}}
for mod in cfg_data.get("modules", {}):
    py = Path(MODULE_DIR) / f"{mod}.py"
    status = f"✓  ({py.stat().st_size:,} bytes)" if py.exists() else "✗  MISSING"
    print(f"  {mod:25s}  {status}")

# Taxonomy JSONs
print()
print("Taxonomy JSONs:")
for mod, info in cfg_data.get("modules", {}).items():
    tf = info.get("taxonomy_file", "")
    if not tf:
        print(f"  {mod:25s}  (no taxonomy)")
        continue
    p = Path(TAXONOMY_DIR) / tf
    if p.exists():
        tax  = json.load(open(p))
        n_l1 = len(tax)
        n_lf = sum(len(v.get("children",{})) for v in tax.values())
        print(f"  {tf:35s}  {n_l1} L1, {n_lf} leaf")
    else:
        print(f"  {tf:35s}  ✗ MISSING")

# Enhancement modules
print()
print("Enhancement modules:")
for mod_file in ["confidence.py", "feedback.py", "hierarchical.py",
                 "incremental.py", "cost_tracker.py", "language.py",
                 "quality_dashboard.py"]:
    p = Path(mod_file)
    status = f"✓  ({p.stat().st_size:,} bytes)" if p.exists() else "✗  not found"
    print(f"  {mod_file:30s}  {status}")

## 4 · Stale module check

In [ ]:
manifest = json.load(open(MANIFEST_FILE)) if Path(MANIFEST_FILE).exists() else {"modules":{}}
stale = [m for m, info in manifest.get("modules", {}).items()
         if "py_hash" not in info]

if stale:
    print(f"⚠️  Stale .py files detected: {stale}")
    print("   Re-run with --force to update:")
    for m in stale:
        print(f"     rca build --force {m} --input {DEFINITION_FILE}")
else:
    print("✓ All module .py files are up to date with the Excel definition")

## 5 · Discovery cold-start *(optional)*

Run this section **only if** you have no taxonomy files yet.
Set `RUN_DISCOVERY = True` and configure which modules to discover.

In [ ]:
# ── Discovery config ───────────────────────────────────────────
RUN_DISCOVERY = True    # ← set True to run

DISCOVER_MODULES = ["customer_effort"]
SAMPLE_N  = 500
TOP_N     = 80
BATCH_SIZE = 10

In [ ]:
if RUN_DISCOVERY:
    result = subprocess.run(
        [sys.executable, "-m", "rca_pipeline.cli", "discover",
         "--input",        DEFINITION_FILE,
         "--data",         DATA_FILE,
         "--taxonomy-dir", TAXONOMY_DIR,
         "--sample-n",     str(SAMPLE_N),
         "--top-n",        str(TOP_N),
         "--batch-size",   str(BATCH_SIZE),
         "--model",        MODEL,
         "--concurrency",  str(CONCURRENCY),
         "--retries",      str(RETRIES),
         "--id-col",       ID_COL,
         "--text-col",     TEXT_COL,
         ] + (["--modules"] + DISCOVER_MODULES if DISCOVER_MODULES else []),
        capture_output=True, text=True, encoding="utf-8", errors="replace", cwd="."
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
else:
    print("RUN_DISCOVERY = False — skipping discovery phase")

## 6 · Review & edit drafts *(if discovery was run)*

In [ ]:
draft_dir = Path(TAXONOMY_DIR) / "drafts"

if draft_dir.exists():
    drafts = sorted(draft_dir.glob("*_draft.json"))
    if drafts:
        print(f"Draft files in {draft_dir}/:")
        for d in drafts:
            tax  = json.load(open(d))
            n_l1 = len(tax)
            n_lf = sum(len(v.get("children",{})) for v in tax.values())
            print(f"  {d.name:40s}  {n_l1} L1 groups, {n_lf} leaf codes")
        print()
        print("Review and edit the draft files, then run Section 7 to promote.")
    else:
        print("No draft files found. Run discovery first (Section 5).")
else:
    print("No drafts directory — discovery has not been run yet.")

## 7 · Promote drafts to live *(after review)*

In [ ]:
PROMOTE_DRAFTS = True   # ← set True after reviewing drafts

if PROMOTE_DRAFTS:
    result = subprocess.run(
        [sys.executable, "-m", "rca_pipeline.cli", "promote-drafts",
         "--input",        DEFINITION_FILE,
         "--taxonomy-dir", TAXONOMY_DIR,
         "--manifest-file",MANIFEST_FILE,
        ],
        capture_output=True, text=True, encoding="utf-8", errors="replace", cwd="."
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
else:
    print("PROMOTE_DRAFTS = False — skipping.")

## 8 · Feedback store setup *(Enhancement 2)*

Initialise the feedback directory and review any existing human corrections.
These corrections will be injected as few-shot examples during classification.

In [ ]:
Path(FEEDBACK_DIR).mkdir(parents=True, exist_ok=True)

try:
    from rca_pipeline.enhancements.feedback import FeedbackStore

    store = FeedbackStore(FEEDBACK_DIR)
    stats = store.get_stats()

    if stats:
        print("Existing feedback corrections:")
        for module, count in sorted(stats.items()):
            print(f"  {module:25s}  {count:4d} corrections")
            prompt = store.build_few_shot_prompt(module, max_examples=3)
            if prompt:
                print(f"    → {len(prompt)} chars of few-shot prompt will be injected")
    else:
        print("No corrections stored yet — the feedback store is empty.")
        print("After classification, review low-confidence records and add corrections.")
    print()
    print(f"Feedback directory: {Path(FEEDBACK_DIR).resolve()}")
except ImportError:
    print("⚠️  feedback.py not found — few-shot feedback disabled")

## 9 · Import corrections from reviewed spreadsheet *(optional)*

If you have a reviewed spreadsheet with `_corrected` columns, import them here.

In [ ]:
IMPORT_CORRECTIONS = False   # ← set True to import
REVIEW_FILE = "runs/output/review_queue_corrected.xlsx"
IMPORT_MODULE = "contact_reason"

if IMPORT_CORRECTIONS:
    try:
        from rca_pipeline.enhancements.feedback import FeedbackStore
        store = FeedbackStore(FEEDBACK_DIR)
        count = store.import_from_review_file(
            REVIEW_FILE, module=IMPORT_MODULE,
            text_col=TEXT_COL, uid_col=ID_COL,
        )
        print(f"Imported {count} corrections from {REVIEW_FILE}")
    except FileNotFoundError:
        print(f"File not found: {REVIEW_FILE}")
    except ImportError:
        print("⚠️  feedback.py not found")
else:
    print("IMPORT_CORRECTIONS = False — skipping import.")

## 10 · Final readiness check

In [ ]:
print("=== Pipeline readiness check ===\n")

cfg_data = json.load(open(CONFIG_FILE)) if Path(CONFIG_FILE).exists() else {"modules":{}}
all_ready = True

for mod, info in cfg_data.get("modules", {}).items():
    py_ok  = (Path(MODULE_DIR) / f"{mod}.py").exists()
    tf     = info.get("taxonomy_file", "")
    tax_ok = (not tf) or (Path(TAXONOMY_DIR) / tf).exists()

    status = "✓" if py_ok and tax_ok else "✗"
    issues = []
    if not py_ok:  issues.append("missing .py")
    if not tax_ok: issues.append("missing taxonomy JSON")
    if issues:     all_ready = False
    print(f"  {status} {mod:25s}  {', '.join(issues) if issues else 'ready'}")

# Enhancement modules
print()
print("Enhancement modules:")
enhancements = {
    "confidence.py": "Confidence scoring",
    "feedback.py": "Human-in-the-loop feedback",
    "hierarchical.py": "Two-step classification",
    "incremental.py": "Incremental processing",
    "cost_tracker.py": "Cost tracking",
    "language.py": "Language detection",
    "quality_dashboard.py": "Quality dashboard",
}
for f, desc in enhancements.items():
    ok = Path(f).exists()
    print(f"  {'✓' if ok else '○'} {desc:30s}  ({f})")

print()
if all_ready:
    print("All modules ready → open 02_classification.ipynb to classify your data.")
else:
    print("Fix issues above, then re-run rca build.")

In [ ]:
# Force-reload modules after rca build regenerated .py files
import importlib

modules_to_reload = []
cfg_path = Path(CONFIG_FILE)
if cfg_path.exists():
    cfg_data = json.load(open(cfg_path))
    modules_to_reload = list(cfg_data.get("modules", {}).keys())

reloaded = []
for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        try:
            importlib.reload(sys.modules[mod_name])
            reloaded.append(mod_name)
        except Exception as e:
            print(f"  Could not reload {mod_name}: {e}")

if "pipeline" in sys.modules:
    import pipeline as _pl
    # Force re-execution of module-level code on reload
    importlib.reload(_pl)
    from rca_pipeline.pipeline import MODULE_REGISTRY
    print(f"MODULE_REGISTRY reloaded: {list(MODULE_REGISTRY.keys())}")

print(f"Reloaded: {reloaded}")
print("\nRun 02_classification.ipynb now.")